In [3]:
!pip install -q duckdb


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import duckdb
import pandas as pd

In [14]:
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"


months = [('2025', '01'), ('2025', '02'), ('2025', '03'), ('2025', '04'), ('2025', '05'), ('2025', '06')]
# months = [
#     ('2025', '01'), ('2025', '02'), ('2025', '03'), ('2025', '04'),
#     ('2025', '07'), ('2025', '08'),
#     ('2025', '09'), ('2025', '10'), ('2025', '11'), ('2025', '12'),
#     ('2026', '01'), ('2026', '02'), ('2026', '03'), ('2026', '04'),
# ]

parquet_urls = [f"{BASE_URL}/fhvhv_tripdata_{year}-{month}.parquet" for year, month in months]
print(f"Всего файлов: {len(parquet_urls)}")
for u in parquet_urls[:3]:
    print(" -", u)

# --- Подключаемся к DuckDB и работаем НАПРЯМУЮ по HTTP, без скачивания на диск ---
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")  # расширение для чтения файлов по HTTP

parquet_str = "', '".join(parquet_urls)
con.execute(f"""
    CREATE OR REPLACE VIEW trips AS
    SELECT * FROM read_parquet(['{parquet_str}'])
""")

company_stats = con.execute("""
    SELECT
        hvfhs_license_num,
        COUNT(*) as trips_count,
        AVG(trip_miles) as avg_miles,
        AVG(base_passenger_fare) as avg_fare,
        AVG(tips) as avg_tip
    FROM trips
    GROUP BY hvfhs_license_num
    ORDER BY trips_count DESC
""").df()

print(company_stats)


Всего файлов: 6
 - https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-01.parquet
 - https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-02.parquet
 - https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-03.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

HTTPException: HTTP Error: HTTP GET error on 'https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-06.parquet' (HTTP 403 Forbidden)